In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path


def find_repository_root(start=Path.cwd()):
    """Find the repository root from Jupyter's current working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from inside "
        "NYC_Healthcare_Accessibility."
    )


REPO_ROOT = find_repository_root()
INPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_inputs"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Libraries loaded.")

Libraries loaded.


In [3]:
df = pd.read_csv(INPUT_DIR / "Brooklyn! - Brooklyn_2100_INDICATORS_6.csv")
df.head(1208)

,Unnamed: 0,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,1,o_360470001001,1740.890698,1452.612,0,26.166667,24.700000,1.816667
1,2,o_360470001002,1523.935634,1345.087,0,23.966667,22.766667,4.150000
2,3,o_360470001003,2286.113270,1298.783,0,24.633333,22.133333,4.766667
3,4,o_360470001004,1673.343414,1091.553,0,19.466667,18.466667,2.400000
4,5,o_360470002001,1967.597636,1419.352,0,25.433333,23.933333,1.616667
...,...,...,...,...,...,...,...,...
1203,1204,o_360470480002,2702.355494,784.392,0,23.716667,13.283333,2.216667
1204,1205,o_360470481001,2146.094781,1345.730,0,26.716667,22.850000,2.083333
1205,1206,o_360470481002,1942.342657,1459.390,0,27.033333,24.700000,1.766667
1206,1207,o_360470481003,1945.861657,1462.909,0,27.066667,24.733333,1.733333


In [4]:
print(df.shape)
print(df.columns.tolist())

(2112, 8)
['Unnamed: 0', 'from_id', 'total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [5]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

df["GEOID_TEXT"] = df["from_id"].astype(str).str.replace("o_", "", regex=False)

df[["from_id", "GEOID_TEXT"]].head(1208)

,from_id,GEOID_TEXT
0,o_360470001001,360470001001
1,o_360470001002,360470001002
2,o_360470001003,360470001003
3,o_360470001004,360470001004
4,o_360470002001,360470002001
...,...,...
1203,o_360470480002,360470480002
1204,o_360470481001,360470481001
1205,o_360470481002,360470481002
1206,o_360470481003,360470481003


In [6]:
benefit_cols = []
cost_cols = ["total_distance",

    "walking_distance",

    "transfers",

    "travel_time_total",

    "walking_time",


    "wait_time_total"]

criteria_cols = benefit_cols + cost_cols

print("Benefit indicators, higher is better:")
print(benefit_cols)

print("\nCost indicators, lower is better:")
print(cost_cols)

print("\nAll criteria:")
print(criteria_cols)

Benefit indicators, higher is better:
[]

Cost indicators, lower is better:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']

All criteria:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [7]:
min_max_table = pd.DataFrame({
    "min": df[criteria_cols].min(),
    "max": df[criteria_cols].max()
})

print("Min and max for each indicator:")
display(min_max_table)

Min and max for each indicator:


,min,max
total_distance,307.234615,13259.949770
walking_distance,140.982000,3631.541000
transfers,0.000000,2.000000
travel_time_total,3.616667,78.366667
walking_time,2.416667,61.300000
wait_time_total,1.016667,16.483333


In [8]:
# All of our current indicators are cost indicators
# Lower = better, so we use: (max - value) / (max - min)

normalized = pd.DataFrame(index=df.index)

for col in criteria_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    
    normalized[col] = (max_val - df[col]) / (max_val - min_val)

print("Normalized values:")
display(normalized.head())

Normalized values:


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.889316,0.624235,1.0,0.698328,0.621568,0.948276
1,0.906066,0.655039,1.0,0.727759,0.654401,0.797414
2,0.847223,0.668305,1.0,0.718841,0.665157,0.757543
3,0.894531,0.727674,1.0,0.787960,0.727427,0.910560
4,0.871814,0.633764,1.0,0.708138,0.634588,0.961207


In [9]:
r_column_sums = normalized[criteria_cols].sum()

print("Step 2 preparation: Sum of each standardized column")
print("These sums go in the denominator for p_ij.")
display(r_column_sums)

Step 2 preparation: Sum of each standardized column
These sums go in the denominator for p_ij.


total_distance       1752.845597
walking_distance     1480.667363
transfers            1912.000000
travel_time_total    1508.705240
walking_time         1478.552222
wait_time_total      1759.293103
dtype: float64

In [10]:
P = normalized[criteria_cols] / r_column_sums

print("Step 2: Probability matrix p_ij")
print("Each standardized value is divided by its column total.")
display(P.head())

Step 2: Probability matrix p_ij
Each standardized value is divided by its column total.


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.000507,0.000422,0.000523,0.000463,0.000420,0.000539
1,0.000517,0.000442,0.000523,0.000482,0.000443,0.000453
2,0.000483,0.000451,0.000523,0.000476,0.000450,0.000431
3,0.000510,0.000491,0.000523,0.000522,0.000492,0.000518
4,0.000497,0.000428,0.000523,0.000469,0.000429,0.000546


In [11]:
print("values should add up to one for each column, since they are probabilities.")
display(P.sum())

values should add up to one for each column, since they are probabilities.


total_distance       1.0
walking_distance     1.0
transfers            1.0
travel_time_total    1.0
walking_time         1.0
wait_time_total      1.0
dtype: float64

In [12]:
P_safe = P.replace(0, 1e-12)

print("Step 3 preparation: Replace 0 values so ln(0) does not break")
display(P_safe.head())

Step 3 preparation: Replace 0 values so ln(0) does not break


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.000507,0.000422,0.000523,0.000463,0.000420,0.000539
1,0.000517,0.000442,0.000523,0.000482,0.000443,0.000453
2,0.000483,0.000451,0.000523,0.000476,0.000450,0.000431
3,0.000510,0.000491,0.000523,0.000522,0.000492,0.000518
4,0.000497,0.000428,0.000523,0.000469,0.000429,0.000546


In [13]:
ln_P = np.log(P_safe)

print("Step 3: Natural log of p_ij")
display(ln_P.head())

Step 3: Natural log of p_ij


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-7.586298,-7.771477,-7.555905,-7.678074,-7.774329,-7.525777
1,-7.567639,-7.723308,-7.555905,-7.636792,-7.722853,-7.699049
2,-7.634787,-7.703259,-7.555905,-7.649123,-7.706551,-7.750342
3,-7.580451,-7.618151,-7.555905,-7.557315,-7.617060,-7.566362
4,-7.606176,-7.756328,-7.555905,-7.664123,-7.753598,-7.512233


In [14]:
P_ln_P = P_safe * ln_P

print("Step 4: p_ij times ln(p_ij)")
display(P_ln_P.head(8))

Step 4: p_ij times ln(p_ij)


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-0.003849,-0.003276,-0.003952,-0.003554,-0.003268,-0.004056
1,-0.003912,-0.003417,-0.003952,-0.003684,-0.003418,-0.003490
2,-0.003690,-0.003477,-0.003952,-0.003645,-0.003467,-0.003337
3,-0.003869,-0.003744,-0.003952,-0.003947,-0.003747,-0.003916
4,-0.003783,-0.003320,-0.003952,-0.003597,-0.003328,-0.004104
5,-0.003887,-0.003814,-0.003952,-0.003877,-0.003815,-0.003787
6,-0.003910,-0.003917,-0.003952,-0.003959,-0.003920,-0.003444
7,-0.003788,-0.003117,-0.003952,-0.003371,-0.003119,-0.004028


In [15]:
p_ln_p_sums = P_ln_P.sum(axis=0)

print("Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator")
display(p_ln_p_sums)

Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator


total_distance      -7.646155
walking_distance    -7.631254
transfers           -7.617534
travel_time_total   -7.638045
walking_time        -7.631038
wait_time_total     -7.634485
dtype: float64

In [16]:
n = len(normalized)
k = 1 / np.log(n)

entropy = -k * p_ln_p_sums

print("Number of rows:", n)
print("k value:", k)
print("Step 5: Entropy for each indicator")
display(entropy)

Number of rows: 2112
k value: 0.1306269067635162
Step 5: Entropy for each indicator


total_distance       0.998794
walking_distance     0.996847
transfers            0.995055
travel_time_total    0.997734
walking_time         0.996819
wait_time_total      0.997269
dtype: float64

In [17]:
#df["fare"].value_counts()

In [18]:
# Step 6: Diversity
# Diversity tells us how much useful variation each indicator has

diversity = 1 - entropy

print("Step 6: Diversity for each indicator")
display(diversity)

Step 6: Diversity for each indicator


total_distance       0.001206
walking_distance     0.003153
transfers            0.004945
travel_time_total    0.002266
walking_time         0.003181
wait_time_total      0.002731
dtype: float64

In [19]:
# Step 7: Calculate entropy weights
# Weight = diversity of one indicator / total diversity of all indicators

weights = diversity / diversity.sum()

print("Step 7: Entropy weights for each indicator")
display(weights)

Step 7: Entropy weights for each indicator


total_distance       0.069009
walking_distance     0.180349
transfers            0.282862
travel_time_total    0.129609
walking_time         0.181962
wait_time_total      0.156209
dtype: float64

In [20]:
weights_table = pd.DataFrame({
    "entropy": entropy,
    "diversity": diversity,
    "weight": weights
})

print("Final entropy weight table:")
display(weights_table)

Final entropy weight table:


,entropy,diversity,weight
total_distance,0.998794,0.001206,0.069009
walking_distance,0.996847,0.003153,0.180349
transfers,0.995055,0.004945,0.282862
travel_time_total,0.997734,0.002266,0.129609
walking_time,0.996819,0.003181,0.181962
wait_time_total,0.997269,0.002731,0.156209


In [21]:
display(weights_table.sort_values(by="weight", ascending=False))

,entropy,diversity,weight
transfers,0.995055,0.004945,0.282862
walking_time,0.996819,0.003181,0.181962
walking_distance,0.996847,0.003153,0.180349
wait_time_total,0.997269,0.002731,0.156209
travel_time_total,0.997734,0.002266,0.129609
total_distance,0.998794,0.001206,0.069009


In [22]:
# Step 8: Calculate final EWM accessibility score
# Formula: score for each block group = sum(normalized value * indicator weight)

df["ewm_accessibility_score"] = (normalized[criteria_cols] * weights).sum(axis=1)

print("Step 8: Final EWM accessibility score")
display(df[["from_id", "GEOID_TEXT", "ewm_accessibility_score"]].head(30))

Step 8: Final EWM accessibility score


,from_id,GEOID_TEXT,ewm_accessibility_score
0,o_360470001001,360470001001,0.808554
1,o_360470001002,360470001002,0.801488
2,o_360470001003,360470001003,0.794393
3,o_360470001004,360470001004,0.852556
4,o_360470002001,360470002001,0.814725
5,o_360470003011,360470003011,0.850964
6,o_360470003012,360470003012,0.848215
7,o_360470003013,360470003013,0.788738
8,o_360470003014,360470003014,0.743894
9,o_360470003015,360470003015,0.839304


In [23]:
print("Score summary:")
display(df["ewm_accessibility_score"].describe())

Score summary:


count    2112.000000
mean        0.789882
std         0.107549
min         0.106627
25%         0.736785
50%         0.805345
75%         0.867867
max         0.982981
Name: ewm_accessibility_score, dtype: float64

In [24]:
final_results = df[
    ["GEOID_TEXT", "from_id"] + criteria_cols + ["ewm_accessibility_score"]
].copy()

display(final_results.head(20))

,GEOID_TEXT,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,ewm_accessibility_score
0,360470001001,o_360470001001,1740.890698,1452.612,0,26.166667,24.700000,1.816667,0.808554
1,360470001002,o_360470001002,1523.935634,1345.087,0,23.966667,22.766667,4.150000,0.801488
2,360470001003,o_360470001003,2286.113270,1298.783,0,24.633333,22.133333,4.766667,0.794393
3,360470001004,o_360470001004,1673.343414,1091.553,0,19.466667,18.466667,2.400000,0.852556
4,360470002001,o_360470002001,1967.597636,1419.352,0,25.433333,23.933333,1.616667,0.814725
5,360470003011,o_360470003011,1611.287890,1037.079,0,20.666667,17.583333,2.933333,0.850964
6,360470003012,o_360470003012,1529.937890,955.729,0,19.266667,16.183333,4.333333,0.848215
7,360470003013,o_360470003013,1951.233421,1573.520,0,29.250000,26.616667,1.933333,0.788738
8,360470003014,o_360470003014,2253.551864,1862.733,0,33.883333,31.483333,2.450000,0.743894
9,360470003015,o_360470003015,1260.486890,686.278,0,14.666667,11.583333,8.933333,0.839304


In [25]:
final_results.to_csv(OUTPUT_DIR / "brooklyn_ewm_results.csv", index=False)
weights_table.to_csv(OUTPUT_DIR / "brooklyn_ewm_weights.csv", index=True)

print("Saved brooklyn_ewm_results.csv")
print("Saved brooklyn_ewm_weights.csv")

Saved brooklyn_ewm_results.csv
Saved brooklyn_ewm_weights.csv
